
## Folder 07 — File 05: Gold Shipping Metrics:


This notebook creates **shipping** and **delivery analytics** from **silver_shipments** and **silver_orders**.

It covers:

- Total shipments and shipment status distribution.
- Carrier-wise shipment performance.
- Delivered and undelivered shipments.
- Average delivery duration.
- Minimum and maximum delivery duration.
- Daily shipment trends.
- Delivery rate and shipment completion rate.
- Order-level shipment summary.

**Important distinction:**

- Delivery duration is calculated only for delivered shipments with valid shipment and delivery dates. 
- Undelivered shipments must not be included in the average.

In [0]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

print("Libraries imported successfully.")


# ============================================================
# 2. PROJECT CONFIGURATION
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

SHIPMENTS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_shipments"
)

ORDERS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_orders"
)

GOLD_SHIPPING_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_shipping_metrics"
)

GOLD_CARRIER_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_carrier_performance"
)

GOLD_ORDER_SHIPPING_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_order_shipping_summary"
)

RUN_ID = spark.sql(
    "SELECT uuid() AS run_id"
).first()["run_id"]

RUN_TIMESTAMP = datetime.now()

print("Run ID:", RUN_ID)


# ============================================================
# 3. VALIDATE SOURCE TABLES
# ============================================================

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}"
)

SOURCE_TABLES = [
    SHIPMENTS_TABLE,
    ORDERS_TABLE
]

for table_name in SOURCE_TABLES:

    if not spark.catalog.tableExists(table_name):

        raise Exception(
            f"Missing source table: {table_name}"
        )

print("All source tables are available.")


# ============================================================
# 4. LOAD SILVER DATASETS
# ============================================================

shipments_df = spark.table(SHIPMENTS_TABLE)

orders_df = spark.table(ORDERS_TABLE)

print("Shipment records:", shipments_df.count())
print("Order records:", orders_df.count())


# ============================================================
# 5. VALIDATE REQUIRED COLUMNS
# ============================================================

required_shipment_columns = [
    "shipment_id",
    "order_id",
    "carrier",
    "shipment_status",
    "shipped_date",
    "delivered_date"
]

required_order_columns = [
    "order_id",
    "customer_id",
    "order_timestamp",
    "order_status",
    "currency",
    "order_total"
]

for column_name in required_shipment_columns:

    if column_name not in shipments_df.columns:

        raise Exception(
            f"Missing shipment column: {column_name}"
        )

for column_name in required_order_columns:

    if column_name not in orders_df.columns:

        raise Exception(
            f"Missing order column: {column_name}"
        )


# ============================================================
# 6. CLEAN SHIPMENT DATA
# ============================================================

# IMPORTANT:
# RETURN_TO_ORIGIN and UNKNOWN_STATUS must not be discarded.
#
# RETURN_TO_ORIGIN is counted under returned_shipments.
# UNKNOWN_STATUS is included in total_shipments.

VALID_SHIPMENT_STATUSES = [
    "PENDING",
    "PROCESSING",
    "SHIPPED",
    "IN_TRANSIT",
    "OUT_FOR_DELIVERY",
    "DELIVERED",
    "CANCELLED",
    "RETURNED",
    "RETURN_TO_ORIGIN",
    "UNKNOWN_STATUS"
]

shipments_clean_df = (
    shipments_df

    .withColumn(
        "shipment_id",
        F.trim(F.col("shipment_id"))
    )

    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )

    .withColumn(
        "carrier",
        F.upper(F.trim(F.col("carrier")))
    )

    .withColumn(
        "shipment_status",
        F.upper(F.trim(F.col("shipment_status")))
    )

    .withColumn(
        "shipped_date",
        F.to_timestamp("shipped_date")
    )

    .withColumn(
        "delivered_date",
        F.to_timestamp("delivered_date")
    )

    .filter(
        F.col("shipment_id").isNotNull()
        & (F.col("shipment_id") != "")
        & F.col("order_id").isNotNull()
        & (F.col("order_id") != "")
        & F.col("carrier").isNotNull()
        & (F.col("carrier") != "")
        & F.col("shipment_status").isin(
            VALID_SHIPMENT_STATUSES
        )
    )
)

print(
    "Clean shipment records:",
    shipments_clean_df.count()
)

print("Shipment status distribution:")

display(
    shipments_clean_df
    .groupBy("shipment_status")
    .count()
    .orderBy("shipment_status")
)


# ============================================================
# 7. DEDUPLICATE SHIPMENTS
# ============================================================

if "_ingestion_timestamp" in shipments_clean_df.columns:

    shipment_window = (
        Window
        .partitionBy("shipment_id")
        .orderBy(
            F.col("_ingestion_timestamp").desc_nulls_last()
        )
    )

else:

    shipment_window = (
        Window
        .partitionBy("shipment_id")
        .orderBy(
            F.col("shipped_date").desc_nulls_last()
        )
    )

shipments_deduplicated_df = (
    shipments_clean_df

    .withColumn(
        "_row_number",
        F.row_number().over(shipment_window)
    )

    .filter(
        F.col("_row_number") == 1
    )

    .drop("_row_number")
)

print(
    "Deduplicated shipment records:",
    shipments_deduplicated_df.count()
)


# ============================================================
# 8. VALIDATE SHIPMENT DATES AND DELIVERY DURATION
# ============================================================

shipments_enriched_df = (
    shipments_deduplicated_df

    .withColumn(
        "shipment_date",
        F.to_date("shipped_date")
    )

    .withColumn(
        "delivery_date",
        F.to_date("delivered_date")
    )

    .withColumn(
        "delivery_duration_hours",

        F.when(
            (
                F.col("shipment_status") == "DELIVERED"
            )
            & F.col("shipped_date").isNotNull()
            & F.col("delivered_date").isNotNull()
            & (
                F.col("delivered_date")
                >= F.col("shipped_date")
            ),

            F.round(
                (
                    F.col("delivered_date").cast("long")
                    - F.col("shipped_date").cast("long")
                ) / 3600,
                2
            )
        )
    )

    .withColumn(
        "delivery_duration_days",

        F.when(
            F.col("delivery_duration_hours").isNotNull(),

            F.round(
                F.col("delivery_duration_hours") / 24,
                2
            )
        )
    )

    .withColumn(
        "date_validation_status",

        F.when(
            F.col("shipped_date").isNull(),
            "MISSING_SHIPPED_DATE"
        )

        .when(
            (
                F.col("shipment_status") == "DELIVERED"
            )
            & F.col("delivered_date").isNull(),
            "MISSING_DELIVERED_DATE"
        )

        .when(
            F.col("delivered_date").isNotNull()
            & F.col("shipped_date").isNotNull()
            & (
                F.col("delivered_date")
                < F.col("shipped_date")
            ),
            "INVALID_DATE_SEQUENCE"
        )

        .otherwise("VALID")
    )
)


# ============================================================
# 9. CREATE DAILY SHIPPING METRICS
# ============================================================

# total_shipments includes every recognized shipment status.
#
# RETURNED and RETURN_TO_ORIGIN are grouped together.
#
# UNKNOWN_STATUS is included in total_shipments, but is not
# included in any of the named status-specific columns.

daily_shipping_df = (
    shipments_enriched_df

    .filter(
        F.col("shipment_date").isNotNull()
    )

    .groupBy(
        "shipment_date"
    )

    .agg(

        F.countDistinct("shipment_id").alias(
            "total_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "DELIVERED",
                1
            ).otherwise(0)
        ).alias(
            "delivered_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "SHIPPED",
                1
            ).otherwise(0)
        ).alias(
            "shipped_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "IN_TRANSIT",
                1
            ).otherwise(0)
        ).alias(
            "in_transit_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "OUT_FOR_DELIVERY",
                1
            ).otherwise(0)
        ).alias(
            "out_for_delivery_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "PROCESSING",
                1
            ).otherwise(0)
        ).alias(
            "processing_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "PENDING",
                1
            ).otherwise(0)
        ).alias(
            "pending_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "CANCELLED",
                1
            ).otherwise(0)
        ).alias(
            "cancelled_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status").isin(
                    "RETURNED",
                    "RETURN_TO_ORIGIN"
                ),
                1
            ).otherwise(0)
        ).alias(
            "returned_shipments"
        ),

        F.round(
            F.avg("delivery_duration_hours"),
            2
        ).alias(
            "average_delivery_hours"
        ),

        F.round(
            F.avg("delivery_duration_days"),
            2
        ).alias(
            "average_delivery_days"
        ),

        F.min(
            "delivery_duration_days"
        ).alias(
            "minimum_delivery_days"
        ),

        F.max(
            "delivery_duration_days"
        ).alias(
            "maximum_delivery_days"
        ),

        F.sum(
            F.when(
                F.col("date_validation_status") != "VALID",
                1
            ).otherwise(0)
        ).alias(
            "invalid_shipment_date_records"
        )
    )
)


# ============================================================
# 10. CALCULATE DAILY DELIVERY RATES
# ============================================================

daily_shipping_df = (
    daily_shipping_df

    .withColumn(
        "delivery_rate",

        F.when(
            F.col("total_shipments") > 0,

            F.round(
                F.col("delivered_shipments")
                / F.col("total_shipments") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )

    .withColumn(
        "cancellation_rate",

        F.when(
            F.col("total_shipments") > 0,

            F.round(
                F.col("cancelled_shipments")
                / F.col("total_shipments") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )

    .withColumn(
        "return_rate",

        F.when(
            F.col("total_shipments") > 0,

            F.round(
                F.col("returned_shipments")
                / F.col("total_shipments") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )
)


# ============================================================
# 11. CREATE CARRIER PERFORMANCE METRICS
# ============================================================

carrier_performance_df = (
    shipments_enriched_df

    .groupBy(
        "carrier"
    )

    .agg(

        F.countDistinct("shipment_id").alias(
            "total_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "DELIVERED",
                1
            ).otherwise(0)
        ).alias(
            "delivered_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "IN_TRANSIT",
                1
            ).otherwise(0)
        ).alias(
            "in_transit_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "OUT_FOR_DELIVERY",
                1
            ).otherwise(0)
        ).alias(
            "out_for_delivery_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "CANCELLED",
                1
            ).otherwise(0)
        ).alias(
            "cancelled_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status").isin(
                    "RETURNED",
                    "RETURN_TO_ORIGIN"
                ),
                1
            ).otherwise(0)
        ).alias(
            "returned_shipments"
        ),

        F.round(
            F.avg("delivery_duration_days"),
            2
        ).alias(
            "average_delivery_days"
        ),

        F.min(
            "delivery_duration_days"
        ).alias(
            "minimum_delivery_days"
        ),

        F.max(
            "delivery_duration_days"
        ).alias(
            "maximum_delivery_days"
        )
    )

    .withColumn(
        "delivery_rate",

        F.when(
            F.col("total_shipments") > 0,

            F.round(
                F.col("delivered_shipments")
                / F.col("total_shipments") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )

    .withColumn(
        "carrier_cancellation_rate",

        F.when(
            F.col("total_shipments") > 0,

            F.round(
                F.col("cancelled_shipments")
                / F.col("total_shipments") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )

    .withColumn(
        "carrier_return_rate",

        F.when(
            F.col("total_shipments") > 0,

            F.round(
                F.col("returned_shipments")
                / F.col("total_shipments") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )
)


# ============================================================
# 12. RANK CARRIERS BY DELIVERY PERFORMANCE
# ============================================================

carrier_window = (
    Window
    .orderBy(
        F.col("delivery_rate").desc(),
        F.col("average_delivery_days").asc_nulls_last()
    )
)

carrier_performance_df = (
    carrier_performance_df

    .withColumn(
        "carrier_rank",
        F.dense_rank().over(carrier_window)
    )
)


# ============================================================
# 13. CLEAN AND DEDUPLICATE ORDERS
# ============================================================

orders_clean_df = (
    orders_df

    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )

    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id"))
    )

    .withColumn(
        "order_timestamp",
        F.to_timestamp("order_timestamp")
    )

    .withColumn(
        "order_date",
        F.to_date("order_timestamp")
    )

    .withColumn(
        "order_status",
        F.upper(F.trim(F.col("order_status")))
    )

    .withColumn(
        "currency",
        F.upper(F.trim(F.col("currency")))
    )

    .withColumn(
        "order_total",
        F.col("order_total").cast("decimal(18,2)")
    )

    .filter(
        F.col("order_id").isNotNull()
        & F.col("order_timestamp").isNotNull()
        & F.col("currency").isNotNull()
        & F.col("order_total").isNotNull()
    )
)

if "_ingestion_timestamp" in orders_clean_df.columns:

    order_window = (
        Window
        .partitionBy("order_id")
        .orderBy(
            F.col("_ingestion_timestamp").desc_nulls_last()
        )
    )

else:

    order_window = (
        Window
        .partitionBy("order_id")
        .orderBy(
            F.col("order_timestamp").desc_nulls_last()
        )
    )

orders_deduplicated_df = (
    orders_clean_df

    .withColumn(
        "_row_number",
        F.row_number().over(order_window)
    )

    .filter(
        F.col("_row_number") == 1
    )

    .drop("_row_number")
)


# ============================================================
# 14. AGGREGATE SHIPMENT METRICS BY ORDER
# ============================================================

order_shipment_aggregation_df = (
    shipments_enriched_df

    .groupBy("order_id")

    .agg(

        F.countDistinct("shipment_id").alias(
            "total_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "DELIVERED",
                1
            ).otherwise(0)
        ).alias(
            "delivered_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "IN_TRANSIT",
                1
            ).otherwise(0)
        ).alias(
            "in_transit_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "OUT_FOR_DELIVERY",
                1
            ).otherwise(0)
        ).alias(
            "out_for_delivery_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status") == "CANCELLED",
                1
            ).otherwise(0)
        ).alias(
            "cancelled_shipments"
        ),

        F.sum(
            F.when(
                F.col("shipment_status").isin(
                    "RETURNED",
                    "RETURN_TO_ORIGIN"
                ),
                1
            ).otherwise(0)
        ).alias(
            "returned_shipments"
        ),

        F.max(
            F.when(
                F.col("shipment_status") == "DELIVERED",
                F.col("delivered_date")
            )
        ).alias(
            "latest_delivery_date"
        ),

        F.round(
            F.avg("delivery_duration_days"),
            2
        ).alias(
            "average_delivery_days"
        )
    )
)


# ============================================================
# 15. BUILD ORDER-LEVEL SHIPPING SUMMARY
# ============================================================

order_shipping_summary_df = (
    orders_deduplicated_df.alias("orders")

    .join(
        order_shipment_aggregation_df.alias("shipments"),

        F.col("orders.order_id")
        == F.col("shipments.order_id"),

        "left"
    )

    .select(

        F.col("orders.order_id"),

        F.col("orders.customer_id"),

        F.col("orders.order_date"),

        F.col("orders.order_status"),

        F.col("orders.currency"),

        F.col("orders.order_total"),

        F.coalesce(
            F.col("shipments.total_shipments"),
            F.lit(0)
        ).alias(
            "total_shipments"
        ),

        F.coalesce(
            F.col("shipments.delivered_shipments"),
            F.lit(0)
        ).alias(
            "delivered_shipments"
        ),

        F.coalesce(
            F.col("shipments.in_transit_shipments"),
            F.lit(0)
        ).alias(
            "in_transit_shipments"
        ),

        F.coalesce(
            F.col("shipments.out_for_delivery_shipments"),
            F.lit(0)
        ).alias(
            "out_for_delivery_shipments"
        ),

        F.coalesce(
            F.col("shipments.cancelled_shipments"),
            F.lit(0)
        ).alias(
            "cancelled_shipments"
        ),

        F.coalesce(
            F.col("shipments.returned_shipments"),
            F.lit(0)
        ).alias(
            "returned_shipments"
        ),

        F.col("shipments.latest_delivery_date"),

        F.col("shipments.average_delivery_days")
    )

    .withColumn(
        "order_shipping_status",

        F.when(
            F.col("total_shipments") == 0,
            "NO_SHIPMENT"
        )

        .when(
            F.col("delivered_shipments")
            == F.col("total_shipments"),
            "ALL_SHIPMENTS_DELIVERED"
        )

        .when(
            F.col("delivered_shipments") > 0,
            "PARTIALLY_DELIVERED"
        )

        .when(
            F.col("in_transit_shipments")
            + F.col("out_for_delivery_shipments") > 0,
            "IN_TRANSIT"
        )

        .when(
            F.col("cancelled_shipments")
            == F.col("total_shipments"),
            "ALL_SHIPMENTS_CANCELLED"
        )

        .when(
            F.col("returned_shipments") > 0,
            "RETURNED"
        )

        .otherwise("PROCESSING_OR_PENDING")
    )
)


# ============================================================
# 16. ADD AUDIT COLUMNS
# ============================================================

daily_shipping_df = (
    daily_shipping_df

    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )

    .withColumn(
        "created_at",
        F.lit(RUN_TIMESTAMP)
    )
)

carrier_performance_df = (
    carrier_performance_df

    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )

    .withColumn(
        "created_at",
        F.lit(RUN_TIMESTAMP)
    )
)

order_shipping_summary_df = (
    order_shipping_summary_df

    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )

    .withColumn(
        "created_at",
        F.lit(RUN_TIMESTAMP)
    )
)


# ============================================================
# 17. DISPLAY DAILY SHIPPING METRICS
# ============================================================

display(
    daily_shipping_df.orderBy(
        F.col("shipment_date").desc()
    )
)


# ============================================================
# 18. DISPLAY CARRIER PERFORMANCE
# ============================================================

display(
    carrier_performance_df.orderBy(
        F.col("carrier_rank")
    )
)


# ============================================================
# 19. DISPLAY ORDER SHIPPING SUMMARY
# ============================================================

display(
    order_shipping_summary_df.orderBy(
        F.col("order_date").desc()
    )
)


# ============================================================
# 20. SAVE GOLD DAILY SHIPPING METRICS
# ============================================================

(
    daily_shipping_df.write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        GOLD_SHIPPING_TABLE
    )
)

print(
    "Daily shipping table created:",
    GOLD_SHIPPING_TABLE
)


# ============================================================
# 21. SAVE GOLD CARRIER PERFORMANCE
# ============================================================

(
    carrier_performance_df.write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        GOLD_CARRIER_TABLE
    )
)

print(
    "Carrier performance table created:",
    GOLD_CARRIER_TABLE
)


# ============================================================
# 22. SAVE GOLD ORDER SHIPPING SUMMARY
# ============================================================

(
    order_shipping_summary_df.write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        GOLD_ORDER_SHIPPING_TABLE
    )
)

print(
    "Order shipping table created:",
    GOLD_ORDER_SHIPPING_TABLE
)


# ============================================================
# 23. FINAL VALIDATION
# ============================================================

gold_shipping_df = spark.table(
    GOLD_SHIPPING_TABLE
)

gold_carrier_df = spark.table(
    GOLD_CARRIER_TABLE
)

gold_order_shipping_df = spark.table(
    GOLD_ORDER_SHIPPING_TABLE
)

print(
    "Daily shipping records:",
    gold_shipping_df.count()
)

print(
    "Carrier performance records:",
    gold_carrier_df.count()
)

print(
    "Order shipping records:",
    gold_order_shipping_df.count()
)


# ============================================================
# 24. SOURCE VS GOLD RECONCILIATION
# ============================================================

source_valid_date_count = (
    shipments_enriched_df

    .filter(
        F.col("shipment_date").isNotNull()
    )

    .select("shipment_id")

    .distinct()

    .count()
)

gold_total_count = (
    gold_shipping_df

    .agg(
        F.sum("total_shipments").alias("total")
    )

    .first()["total"]
)

gold_returned_count = (
    gold_shipping_df

    .agg(
        F.sum("returned_shipments").alias("total")
    )

    .first()["total"]
)

source_returned_count = (
    shipments_enriched_df

    .filter(
        F.col("shipment_status").isin(
            "RETURNED",
            "RETURN_TO_ORIGIN"
        )
        & F.col("shipment_date").isNotNull()
    )

    .select("shipment_id")

    .distinct()

    .count()
)

source_unknown_count = (
    shipments_enriched_df

    .filter(
        (F.col("shipment_status") == "UNKNOWN_STATUS")
        & F.col("shipment_date").isNotNull()
    )

    .select("shipment_id")

    .distinct()

    .count()
)

print("=" * 65)
print("SOURCE VS GOLD SHIPPING RECONCILIATION")
print("=" * 65)

print(
    "Source shipments with valid dates:",
    source_valid_date_count
)

print(
    "Gold total shipments:",
    gold_total_count
)

print(
    "Source returned shipments:",
    source_returned_count
)

print(
    "Gold returned shipments:",
    gold_returned_count
)

print(
    "Source unknown-status shipments:",
    source_unknown_count
)

print(
    "Shipment total difference:",
    source_valid_date_count - (gold_total_count or 0)
)

if source_valid_date_count != (gold_total_count or 0):

    raise Exception(
        "SHIPPING RECONCILIATION FAILED: "
        "Source and Gold shipment totals do not match."
    )

if source_returned_count != (gold_returned_count or 0):

    raise Exception(
        "SHIPPING RECONCILIATION FAILED: "
        "Source and Gold returned shipment totals do not match."
    )

print("Source and Gold shipment totals reconciled successfully.")


# ============================================================
# 25. FINAL SUMMARY
# ============================================================

print("=" * 65)
print("GOLD SHIPPING METRICS COMPLETED SUCCESSFULLY")
print("=" * 65)

print("Run ID:", RUN_ID)

print("Daily shipping table:", GOLD_SHIPPING_TABLE)

print("Carrier performance table:", GOLD_CARRIER_TABLE)

print("Order shipping table:", GOLD_ORDER_SHIPPING_TABLE)

print("Source shipment total:", source_valid_date_count)

print("Gold shipment total:", gold_total_count)

print("Returned shipments:", gold_returned_count)

print("Unknown-status shipments:", source_unknown_count)

print("=" * 65)

In [0]:
%sql
SELECT
    SUM(total_shipments) AS total_shipments,
    SUM(delivered_shipments) AS delivered_shipments,
    SUM(returned_shipments) AS returned_shipments,
    SUM(cancelled_shipments) AS cancelled_shipments
FROM workspace.ecommerce_gold.gold_shipping_metrics;


## Databricks SQL dashboard queries:


### Daily delivery performance

In [0]:
%sql

SELECT
    shipment_date,
    total_shipments,
    delivered_shipments,
    in_transit_shipments,
    cancelled_shipments,
    returned_shipments,
    delivery_rate,
    average_delivery_days
FROM workspace.ecommerce_gold.gold_shipping_metrics
ORDER BY shipment_date;


### Carrier performance ranking

In [0]:
%sql

SELECT
    carrier_rank,
    carrier,
    total_shipments,
    delivered_shipments,
    delivery_rate,
    average_delivery_days,
    carrier_cancellation_rate,
    carrier_return_rate
FROM workspace.ecommerce_gold.gold_carrier_performance
ORDER BY carrier_rank;


### Orders without shipments

In [0]:
%sql

SELECT
    order_id,
    customer_id,
    order_date,
    order_status,
    order_total,
    order_shipping_status
FROM workspace.ecommerce_gold.gold_order_shipping_summary
WHERE order_shipping_status = 'NO_SHIPMENT';


### Orders with incomplete delivery

In [0]:
%sql

SELECT
    order_id,
    total_shipments,
    delivered_shipments,
    in_transit_shipments,
    returned_shipments,
    order_shipping_status
FROM workspace.ecommerce_gold.gold_order_shipping_summary
WHERE order_shipping_status NOT IN (
    'ALL_SHIPMENTS_DELIVERED',
    'ALL_SHIPMENTS_CANCELLED'
);

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.gold_shipping_metrics;